> **TÍTULO:** Cyber Sentinel
>
> **DATASET:** [UNSW-NB15](https://research.unsw.edu.au/projects/unsw-nb15-dataset) (Training-set / Testing-set)


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)

np.random.seed(42)

In [3]:
df_train = pd.read_csv("data/UNSW_NB15_training-set.csv")
df_test  = pd.read_csv("data/UNSW_NB15_testing-set.csv")

# Marcamos a origem de cada registro antes de combinar, para podermos separar novamente se necessário
df_train["origem"] = "train"
df_test["origem"]  = "test"

df_full = pd.concat([df_train, df_test], axis=0, ignore_index=True)

print(f"Treino : {df_train.shape[0]:,} linhas x {df_train.shape[1]} colunas".replace(",", "."))
print(f"Teste  : {df_test.shape[0]:,} linhas x {df_test.shape[1]} colunas".replace(",", "."))
print(f"Total  : {df_full.shape[0]:,} linhas x {df_full.shape[1]} colunas".replace(",", "."))

Treino : 175.341 linhas x 46 colunas
Teste  : 82.332 linhas x 46 colunas
Total  : 257.673 linhas x 46 colunas


In [4]:
df_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 257673 entries, 0 to 257672
Data columns (total 46 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   id                 257673 non-null  int64  
 1   dur                257673 non-null  float64
 2   proto              257673 non-null  object 
 3   service            257673 non-null  object 
 4   state              257673 non-null  object 
 5   spkts              257673 non-null  int64  
 6   dpkts              257673 non-null  int64  
 7   sbytes             257673 non-null  int64  
 8   dbytes             257673 non-null  int64  
 9   rate               257673 non-null  float64
 10  sttl               257673 non-null  int64  
 11  dttl               257673 non-null  int64  
 12  sload              257673 non-null  float64
 13  dload              257673 non-null  float64
 14  sloss              257673 non-null  int64  
 15  dloss              257673 non-null  int64  
 16  si

### Principais campos do dataset

| Grupo | Exemplos de campos | Descrição |
|---|---|---|
| Identificação | `id` | Identificador único do registro |
| Rede / Protocolo | `proto`, `service`, `state` | Protocolo de transporte, serviço de aplicação, estado da conexão |
| Volume de tráfego | `spkts`, `dpkts`, `sbytes`, `dbytes`, `rate` | Pacotes/bytes enviados (`s`) e recebidos (`d`), taxa de transmissão |
| Tempo / Latência | `dur`, `sinpkt`, `dinpkt`, `sjit`, `djit`, `tcprtt`, `synack`, `ackdat` | Duração da conexão, jitter, tempos de handshake TCP |
| Contadores de conexão (`ct_*`) | `ct_srv_src`, `ct_state_ttl`, `ct_dst_ltm`, ... | Contagens de conexões similares numa janela de tempo. Usados na correlação temporal do Cyber Sentinel |
| Rótulos | `attack_cat`, `label` | Categoria do ataque (10 classes, incluindo `Normal`) e rótulo binário (0 = normal, 1 = ataque) |

In [9]:
# Separando os campos categóricos (`object`) e numéricos automaticamente, para conferência.
colunas_categoricas = df_full.select_dtypes(include="object").columns.tolist()
colunas_numericas   = df_full.select_dtypes(include=[np.number]).columns.tolist()

print(f"Campos categóricos ({len(colunas_categoricas)}):\n{colunas_categoricas}\n")
print(f"Campos numéricos ({len(colunas_numericas)}):\n{colunas_numericas}")

Campos categóricos (5):
['proto', 'service', 'state', 'attack_cat', 'origem']

Campos numéricos (41):
['id', 'dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports', 'label']


In [10]:
# df.sample() mostram exemplos reais de linhas, permitindo inspecionar visualmente o formato dos dados
df_full.sample(5, random_state=42)

,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,dttl,sload,dload,sloss,dloss,sinpkt,dinpkt,sjit,djit,swin,stcpb,dtcpb,dwin,tcprtt,synack,ackdat,smean,dmean,trans_depth,response_body_len,ct_srv_src,ct_state_ttl,ct_dst_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label,origem
122637,122638,0.000005,udp,dns,INT,2,0,114,0,200000.005100,254,0,9.120000e+07,0.000000,0,0,0.005000,0.000000,0.000000,0.000000,0,0,0,0,0.000000,0.000000,0.000000,57,0,0,0,13,2,9,9,9,13,0,0,0,9,13,0,Generic,1,train
137118,137119,0.000003,udp,dns,INT,2,0,114,0,333333.321500,254,0,1.520000e+08,0.000000,0,0,0.003000,0.000000,0.000000,0.000000,0,0,0,0,0.000000,0.000000,0.000000,57,0,0,0,16,2,16,16,16,16,0,0,0,17,16,0,Generic,1,train
245289,69949,1.167584,tcp,-,CON,6,2,1012,86,5.995286,62,252,5.782882e+03,294.625488,2,1,233.516800,0.001000,13675.729750,0.000000,255,1497222707,3407537974,255,0.940009,0.560296,0.379713,169,43,0,0,2,3,1,1,1,2,0,0,0,2,2,0,Normal,0,test
221262,45922,0.486485,tcp,http,FIN,10,8,798,1568,34.944551,62,252,1.182359e+04,22561.845700,2,2,54.053889,59.958430,2705.531092,82.549047,255,101844950,785426226,255,0.138611,0.065983,0.072628,80,196,1,372,1,1,1,1,1,1,0,0,1,1,1,0,Exploits,1,test
31408,31409,0.072147,udp,-,CON,4,4,568,312,97.024136,31,29,4.723689e+04,25947.025390,0,0,18.674334,23.547666,26.397475,33.299309,0,0,0,0,0.000000,0.000000,0.000000,142,78,0,0,7,0,5,1,1,5,0,0,0,2,8,0,Normal,0,train


In [ ]:
# Amostra de valores únicos para cada campo categórico relevante:
for col in ["proto", "service", "state", "attack_cat"]:
    valores = df_full[col].unique()
    print(f"{col} ({df_full[col].nunique()} valores únicos): {sorted(map(str, valores))[:15]}"
          f"{' ...' if df_full[col].nunique() > 15 else ''}")

proto (133 valores únicos): ['3pc', 'a/n', 'aes-sp3-d', 'any', 'argus', 'aris', 'arp', 'ax.25', 'bbn-rcc', 'bna', 'br-sat-mon', 'cbt', 'cftp', 'chaos', 'compaq-peer'] ...
service (13 valores únicos): ['-', 'dhcp', 'dns', 'ftp', 'ftp-data', 'http', 'irc', 'pop3', 'radius', 'smtp', 'snmp', 'ssh', 'ssl']
state (11 valores únicos): ['ACC', 'CLO', 'CON', 'ECO', 'FIN', 'INT', 'PAR', 'REQ', 'RST', 'URN', 'no']
attack_cat (10 valores únicos): ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Normal', 'Reconnaissance', 'Shellcode', 'Worms']


In [12]:
resumo_numerico = df_full[colunas_numericas].describe().T
resumo_numerico["range"] = resumo_numerico["max"] - resumo_numerico["min"]
resumo_numerico[["min", "max", "mean", "std", "range"]].sort_values("range", ascending=False).head(15)

,min,max,mean,std,range
sload,0.0,5.988000e+09,7.060869e+07,1.857313e+08,5.988000e+09
stcpb,0.0,4.294959e+09,1.006120e+09,1.367795e+09,4.294959e+09
dtcpb,0.0,4.294882e+09,1.002295e+09,1.363877e+09,4.294882e+09
dload,0.0,2.242273e+07,6.582143e+05,2.412372e+06,2.242273e+07
dbytes,0.0,1.465753e+07,1.438729e+04,1.461993e+05,1.465753e+07
sbytes,24.0,1.435577e+07,8.572952e+03,1.737739e+05,1.435575e+07
response_body_len,0.0,6.558056e+06,1.968900e+03,4.962523e+04,6.558056e+06
sjit,0.0,1.483831e+06,5.419373e+03,4.903450e+04,1.483831e+06
rate,0.0,1.000000e+06,9.125391e+04,1.603446e+05,1.000000e+06
djit,0.0,4.631992e+05,5.822515e+02,3.930153e+03,4.631992e+05


In [13]:
# Min/máx específicos dos campos mais relevantes para o Cyber Sentinel (volume e tempo de tráfego):
campos_chave = ["dur", "spkts", "dpkts", "sbytes", "dbytes", "rate", "sttl", "dttl"]
df_full[campos_chave].agg(["min", "max", "mean", "median"]).T

,min,max,mean,median
dur,0.0,5.999999e+01,1.246715,0.004285
spkts,1.0,1.064600e+04,19.777144,4.000000
dpkts,0.0,1.101800e+04,18.514703,2.000000
sbytes,24.0,1.435577e+07,8572.951803,528.000000
dbytes,0.0,1.465753e+07,14387.287543,178.000000
rate,0.0,1.000000e+06,91253.912440,2955.664893
sttl,0.0,2.550000e+02,180.000931,254.000000
dttl,0.0,2.540000e+02,84.754957,29.000000


In [14]:
# Verificando balanceamento das classes (label e attack_cat)
print("Distribuição de 'label' (0 = Normal, 1 = Ataque):")
print(df_full["label"].value_counts())
print(f"Proporção de ataques: {df_full['label'].mean():.2%}\n")

print("Distribuição de 'attack_cat':")
print(df_full["attack_cat"].value_counts())

Distribuição de 'label' (0 = Normal, 1 = Ataque):
label
1    164673
0     93000
Name: count, dtype: int64
Proporção de ataques: 63.91%

Distribuição de 'attack_cat':
attack_cat
Normal            93000
Generic           58871
Exploits          44525
Fuzzers           24246
DoS               16353
Reconnaissance    13987
Analysis           2677
Backdoor           2329
Shellcode          1511
Worms               174
Name: count, dtype: int64


In [17]:
# Removendo linhas duplicadas
qtd_duplicadas = df_full.duplicated().sum()
print(f"Linhas duplicadas encontradas: {qtd_duplicadas}")

if qtd_duplicadas > 0:
    df_full = df_full.drop_duplicates().reset_index(drop=True)
    print(f"Duplicadas removidas. Novo shape: {df_full.shape}")
else:
    print("Nenhuma duplicata encontrada.")

Linhas duplicadas encontradas: 0
Nenhuma duplicata encontrada.


### Padronização de campos categóricos

Removemos espaços em branco acidentais e uniformizamos a caixa (lowercase) em `proto`, `service`,
`state` e `attack_cat`, prática comum para evitar que `'TCP'` e `'tcp '` sejam tratados como categorias
diferentes.


In [18]:
for col in ["proto", "service", "state", "attack_cat"]:
    antes = df_full[col].nunique()
    df_full[col] = df_full[col].astype(str).str.strip()
    depois = df_full[col].nunique()
    print(f"{col}: {antes} -> {depois} categorias únicas após strip()")

proto: 133 -> 133 categorias únicas após strip()
service: 13 -> 13 categorias únicas após strip()
state: 11 -> 11 categorias únicas após strip()
attack_cat: 10 -> 10 categorias únicas após strip()


### Verificação de valores ausentes (nulos)

O dataset oficial do UNSW-NB15 **não possui valores nulos** (confirmado abaixo). Os "valores
faltantes" no domínio de rede aparecem tipicamente representados pelo placeholder `'-'` no campo
`service`, não como `NaN`. Ainda assim, fazemos a checagem formal.


In [19]:
nulos_por_coluna = df_full.isnull().sum()
total_nulos = nulos_por_coluna.sum()

print(f"Total de valores nulos no dataset: {total_nulos}")
if total_nulos > 0:
    print(nulos_por_coluna[nulos_por_coluna > 0])
else:
    print("Confirmado: nenhum valor nulo (NaN) real no dataset original.")

qtd_service_desconhecido = (df_full["service"] == "-").sum()
print(f"\nRegistros com service='-' (serviço não identificado): {qtd_service_desconhecido} "
      f"({qtd_service_desconhecido/len(df_full):.2%} do total)")

Total de valores nulos no dataset: 0
Confirmado: nenhum valor nulo (NaN) real no dataset original.

Registros com service='-' (serviço não identificado): 141321 (54.85% do total)


### Simulação de Valores Ausentes e Tratamento

Para atender ao requisito de demonstrar o processo de tratamento de dados ausentes, vamos:

1. Criar uma cópia do DataFrame (`df_sim`) e **injetar propositalmente** valores ausentes (`NaN`) em
   uma amostra aleatória de registros, em campos numéricos e categóricos escolhidos;
2. Demonstrar duas estratégias de tratamento:
   - **Remoção** (`dropna`) para os casos onde a proporção de ausência é baixa e a linha pode ser
     descartada sem grande perda de informação;
   - **Substituição/Imputação** (`fillna`) usando mediana para campos numéricos (robusta a outliers,
     comuns em tráfego de rede) e moda para campos categóricos.


In [20]:
df_sim = df_full.copy()

FRACAO_FALTANTE = 0.03  # 3% dos registros recebem valor ausente em cada campo escolhido
campos_para_simular = {
    "numericos": ["dur", "sbytes", "dbytes", "rate"],
    "categoricos": ["service", "state"],
}

rng = np.random.default_rng(42)

for grupo, campos in campos_para_simular.items():
    for col in campos:
        n = len(df_sim)
        idx_faltantes = rng.choice(n, size=int(n * FRACAO_FALTANTE), replace=False)
        df_sim.loc[idx_faltantes, col] = np.nan

print("Valores ausentes simulados por campo:")
print(df_sim[campos_para_simular["numericos"] + campos_para_simular["categoricos"]].isnull().sum())
print(f"\nTotal de linhas com pelo menos um valor ausente: {df_sim.isnull().any(axis=1).sum()} "
      f"de {len(df_sim)} ({df_sim.isnull().any(axis=1).mean():.2%})")


Valores ausentes simulados por campo:
dur        7730
sbytes     7730
dbytes     7730
rate       7730
service    7730
state      7730
dtype: int64

Total de linhas com pelo menos um valor ausente: 42985 de 257673 (16.68%)


In [21]:
df_sim_drop = df_sim.dropna(subset=campos_para_simular["numericos"]).copy()

print(f"Linhas antes da remoção : {len(df_sim)}")
print(f"Linhas depois da remoção: {len(df_sim_drop)}")
print(f"Linhas descartadas      : {len(df_sim) - len(df_sim_drop)}")


Linhas antes da remoção : 257673
Linhas depois da remoção: 228120
Linhas descartadas      : 29553


In [ ]:
df_sim_fill = df_sim.copy()

for col in campos_para_simular["numericos"]:
    mediana = df_sim_fill[col].median()
    df_sim_fill[col] = df_sim_fill[col].fillna(mediana)
    print(f"{col}: preenchido com a mediana ({mediana:.4f})")

for col in campos_para_simular["categoricos"]:
    moda = df_sim_fill[col].mode()[0]
    df_sim_fill[col] = df_sim_fill[col].fillna(moda)
    print(f"{col}: preenchido com a moda ('{moda}')")

print(f"\nValores ausentes restantes após imputação: {df_sim_fill.isnull().sum().sum()}")

dur: preenchido com a mediana (0.0043)
sbytes: preenchido com a mediana (528.0000)
dbytes: preenchido com a mediana (178.0000)
rate: preenchido com a mediana (2958.5798)
service: preenchido com a moda ('-')
state: preenchido com a moda ('FIN')

Valores ausentes restantes após imputação: 0


### Filtro 1: Somente tráfego malicioso (`label == 1`)

Isola todos os registros classificados como ataque, independentemente da categoria: separa o que é "estranho" do que é tráfego normal.


In [23]:
df_ataques = df_full[df_full["label"] == 1]

print(f"Registros de ataque: {len(df_ataques):,} de {len(df_full):,} ({len(df_ataques)/len(df_full):.2%})".replace(",", "."))
df_ataques.head(3)

Registros de ataque: 164.673 de 257.673 (63.91%)


,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,dttl,sload,dload,sloss,dloss,sinpkt,dinpkt,sjit,djit,swin,stcpb,dtcpb,dwin,tcprtt,synack,ackdat,smean,dmean,trans_depth,response_body_len,ct_srv_src,ct_state_ttl,ct_dst_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label,origem
47911,47912,0.000009,ddp,-,INT,2,0,200,0,111111.1072,254,0,88888888.0,0.0,0,0,0.009,0.0,0.0,0.0,0,0,0,0,0.0,0.0,0.0,100,0,0,0,4,2,1,1,1,4,0,0,0,2,4,0,Backdoor,1,train
47912,47913,0.000009,ipv6-frag,-,INT,2,0,200,0,111111.1072,254,0,88888888.0,0.0,0,0,0.009,0.0,0.0,0.0,0,0,0,0,0.0,0.0,0.0,100,0,0,0,7,2,2,2,2,7,0,0,0,2,7,0,Backdoor,1,train
47913,47914,0.000009,cftp,-,INT,2,0,200,0,111111.1072,254,0,88888888.0,0.0,0,0,0.009,0.0,0.0,0.0,0,0,0,0,0.0,0.0,0.0,100,0,0,0,6,2,1,1,1,6,0,0,0,1,6,0,Backdoor,1,train


### Filtro 2: Somente tráfego normal (`label == 0`)

Filtro usado como baseline de comparação (ex.: para treinar o modelo de detecção de anomalia apenas com tráfego legítimo, técnica comum em Isolation Forest/Autoencoders).


In [24]:
df_normal = df_full[df_full["label"] == 0]

print(f"Registros normais: {len(df_normal):,} de {len(df_full):,} ({len(df_normal)/len(df_full):.2%})".replace(",", "."))

Registros normais: 93.000 de 257.673 (36.09%)


### Filtro 3: Uma família específica de ataque (`attack_cat == 'DoS'`)

Isola apenas os registros da categoria "DoS" (Denial of Service), permitindo estudar o comportamento característico desse tipo de ataque (ex.: alto volume de pacotes em curto intervalo de tempo).


In [25]:
df_dos = df_full[df_full["attack_cat"].str.lower() == "dos"]

print(f"Registros da categoria DoS: {len(df_dos):,}".replace(",", "."))
df_dos[["dur", "spkts", "dpkts", "sbytes", "dbytes", "rate", "proto", "state"]].describe().T[["min", "max", "mean"]]


Registros da categoria DoS: 16.353


,min,max,mean
dur,0.0,5.999841e+01,2.448795
spkts,1.0,8.669000e+03,24.117348
dpkts,0.0,1.101800e+04,21.252308
sbytes,24.0,1.163118e+07,18360.793799
dbytes,0.0,1.465753e+07,21747.133737
rate,0.0,1.000000e+06,146165.897023


### Filtro 4: Protocolo específico (`proto == 'tcp'`)

Filtra apenas conexões que usam o protocolo TCP. Útil para análises específicas de handshake (`tcprtt`, `synack`, `ackdat`) que só fazem sentido para esse protocolo.


In [26]:
df_tcp = df_full[df_full["proto"] == "tcp"]

print(f"Conexões TCP: {len(df_tcp):,} de {len(df_full):,} ({len(df_tcp)/len(df_full):.2%})".replace(",", "."))

Conexões TCP: 123.041 de 257.673 (47.75%)


### Filtro 5: Filtro combinado (regra de investigação)

Simula uma regra de triagem do Cyber Sentinel: conexões **TCP**, com estado **`'FIN'`** (conexão finalizada normalmente) que ainda assim foram rotuladas como **ataque**. Esse tipo de filtro combinado ajuda a identificar ataques "silenciosos" que não derrubam a conexão de forma abrupta (ex.: exfiltração de dados via Exploits/Backdoors).


In [27]:
filtro_investigacao = (
    (df_full["proto"] == "tcp") &
    (df_full["state"] == "FIN") &
    (df_full["label"] == 1)
)
df_investigacao = df_full[filtro_investigacao]

print(f"Conexões TCP/FIN classificadas como ataque: {len(df_investigacao):,}".replace(",", "."))
df_investigacao["attack_cat"].value_counts()

Conexões TCP/FIN classificadas como ataque: 55.817


attack_cat
Exploits          27317
Fuzzers           15461
Reconnaissance     6956
DoS                3237
Generic            1002
Shellcode           749
Analysis            621
Backdoor            323
Worms               151
Name: count, dtype: int64

### Filtro 6: Tráfego de alto volume (possível DoS/exfiltração)

Filtra conexões cujo volume de bytes enviados (`sbytes`) está acima do percentil 99, uma heurística simples para sinalizar tráfego anormalmente volumoso.


In [28]:
limite_p99 = df_full["sbytes"].quantile(0.99)
df_alto_volume = df_full[df_full["sbytes"] > limite_p99]

print(f"Limite do percentil 99 de 'sbytes': {limite_p99:,.0f} bytes".replace(",", "."))
print(f"Conexões acima do limite: {len(df_alto_volume):,}".replace(",", "."))
print("\nDistribuição de attack_cat nesse subconjunto:")
print(df_alto_volume["attack_cat"].value_counts())

Limite do percentil 99 de 'sbytes': 72.584 bytes
Conexões acima do limite: 2.577

Distribuição de attack_cat nesse subconjunto:
attack_cat
Exploits          1654
Fuzzers            318
DoS                292
Normal             270
Generic             22
Reconnaissance      11
Backdoor             6
Analysis             3
Worms                1
Name: count, dtype: int64


### Filtro 7: Serviço não identificado + múltiplas conexões recentes (janela temporal)

Combina o campo `service == '-'` (serviço desconhecido, potencialmente scan/reconhecimento) com `ct_state_ttl` elevado (muitas conexões com o mesmo estado/TTL recentemente).


In [29]:
limite_ct = df_full["ct_state_ttl"].quantile(0.90)

filtro_reconhecimento = (
    (df_full["service"] == "-") &
    (df_full["ct_state_ttl"] >= limite_ct)
)
df_reconhecimento = df_full[filtro_reconhecimento]

print(f"Limite (percentil 90) de 'ct_state_ttl': {limite_ct}")
print(f"Registros suspeitos de reconhecimento/scan: {len(df_reconhecimento):,}".replace(",", "."))
df_reconhecimento["attack_cat"].value_counts()

Limite (percentil 90) de 'ct_state_ttl': 2.0
Registros suspeitos de reconhecimento/scan: 64.944


attack_cat
Exploits          16354
Normal            15414
DoS               12615
Fuzzers            8363
Reconnaissance     6871
Analysis           2048
Backdoor           1948
Shellcode           762
Generic             548
Worms                21
Name: count, dtype: int64